# Homework - 3

In this homework, you will learn:

* How to run and train a Diffusion Model
* How to generate high-resolution images
* How to evaluate a generative model performance

As in the previous homework, we expect you to create configurable code and put all the cells in the chronological order, so your solution can be run sequentially. Also, you must use [Comet](https://www.comet.com/site/) for tracking and provide a Comet Report together with the ipynb notebook in your submission. This time, you will also log image samples in the report.

LLM-generated code/analysis for tasks is strongly discouraged.

Penalties will be applied if these rules are not followed.

## Part 1: Understanding the inference pipeline

**Note:** Before starting the homework, we strongly encourage you to read the [DDPM](https://arxiv.org/abs/2006.11239) paper.



In this part, our primary goal is to understand the components of the diffusion pipeline and how they are connected with each other. To do so, we will start with the `diffusers` library. Our particular focus is on the [Stable Diffusion](https://arxiv.org/abs/2112.10752) model.

Stable Diffusion can be seen as an extension of [Denoising Diffusion Probabilistic Models (DDPM)](https://arxiv.org/abs/2006.11239), with the key distinction that the diffusion process is carried out in a *latent* space rather than directly on image pixels. This is enabled by a [Variational Autoencoder (VAE)](https://arxiv.org/abs/1312.6114), which first compresses an image into a lower-dimensional latent representation. The diffusion process is then applied to this latent, and the final denoised representation is decoded back into image space using the VAE decoder.

In more detail, let $x \in \mathbb{R}^{H \times W \times 3}$ be an image. A Variational Autoencoder (VAE) consists of an encoder $E$ and a decoder $D$:
$$
\mu(x), \sigma(x) = E(x), \qquad z_0 = \mu(x) + \sigma(x) \odot \epsilon, \epsilon \sim \mathcal{N}(0, 1) \qquad \hat{x} = D(z_0),
$$
where $z_0 \in \mathbb{R}^{h \times w \times c}$ is a lower-dimensional latent representation.

Instead of applying diffusion directly to $x$, a Latent Diffusion Model (LDM) performs the forward process in the latent space:
$$
q(z_t \mid z_{t-1}) = \mathcal{N}\big(z_t; \sqrt{1 - \beta_t}\, z_{t-1}, \beta_t I \big),
$$
which implies the closed form:
$$
q(z_t \mid z_0) = \mathcal{N}\big(z_t; \sqrt{\bar{\alpha}_t}\, z_0, (1 - \bar{\alpha}_t) I \big),
$$
where $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

The model learns to reverse this process by predicting the noise $\epsilon$ using a neural network $\theta$:
$$
\epsilon_\theta(z_t, t).
$$
The reverse (denoising) step is then:
$$
p_\theta(z_{t-1} \mid z_t) = \mathcal{N}\big(z_{t-1}; \mu_\theta(z_t, t), \sigma_t^2 I \big),
$$
with
$$
\mu_\theta(z_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( z_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \, \epsilon_\theta(z_t, t) \right).
$$
After iteratively denoising to obtain $z_0$, the final image is reconstructed via the decoder:
$$
z_T \to z_0, \qquad \hat{x} = D(z_0).
$$

**Note:** this whole process can be made conditional by adding a condition $c$ as input for the model $\theta$, i.e., the predicted noise becomes $\epsilon_{\theta}(z_t, t, c)$. If, for example, vanilla unconditional diffusion just generates any images similar to the distribution of the training data, a conditional diffusion can generate images aligned with a text prompt.

Typically, a [UNet](https://arxiv.org/abs/1505.04597) architecture is used for denoising. However, Stable Diffusion augments this with cross-attention layers that incorporate information from a text encoder, allowing the model to be conditioned on a textual prompt. Operating in the latent space significantly reduces computational cost—for example, working with $64 \times 64$ representations instead of $512 \times 512$, while still enabling high-quality image generation.

The `diffusers` library. which is similar in spirit to `transformers` for sequence modeling, is developed by Hugging Face and provides a convenient interface for training and running diffusion models. In practice, using Stable Diffusion can be as simple as importing and running a few lines of code:

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format='retina'

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# we load the model in fp16 to reduce computational costs
pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4", dtype=torch.float16)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = pipe.to(device)

In [ ]:
prompt = "A house build of gold blocks and a money garden"
image = pipe(prompt).images[0] # in PIL format

plt.imshow(image)

Let's unpack the pipeline and run its inference ourselves.

We start with a sampled random noise $z_T \sim \mathcal{N}(0,1)$ living in the latent domain and iteratively denoise it using a UNet with attention blocks. To generate images based on a text description, Stable Diffusion integrates text information using cross-attention layers with text embeddings taken from a pretrained backbone, [CLIP](https://arxiv.org/abs/2103.00020). Once we denoised the latents, we apply VAE decoder to map latents back into the pixel space. See the diagram below:

<p align="left">
<img src="https://raw.githubusercontent.com/patrickvonplaten/scientific_images/master/stable_diffusion.png" alt="sd-pipeline" width="500"/>
</p>

So, to run Stable Diffusion, we need VAE, Text Encoder (CLIP), and a denoising model (UNet). Let's load them separately using the `subfolder` option:

In [ ]:
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel, PNDMScheduler

vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device)

# We also load tokenizer as a preprocessor for CLIP
tokenizer = CLIPTokenizer.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="text_encoder").to(device)

unet = UNet2DConditionModel.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="unet").to(device)

In the lecture, we discussed that the Denoising Score Matching view of diffusion uses many different noise levels $\sigma$. The reason is that for small $\sigma$, the score $\nabla \log p_\sigma(x)$ is only accurate locally. If we only trained at small noise levels, we would not be able to start from a random point, since such points lie far from the data distribution and the model would not know how to move these points toward realistic images.

Using many $\sigma$ values allows us to gradually transition from highly noisy samples to clean images. At large $\sigma$, the distribution is smooth and the score provides coarse, global guidance. As $\sigma$ decreases, the model refines the sample with increasingly detailed, local corrections.

To learn this progression, the model $s_{\theta}(x, \sigma)$ must understand how the score changes with $\sigma$. This requires fine-grained supervision across the noise levels. In practice, we therefore train with many $\sigma$ values (e.g., 1000), which results in a dense discretization and a smooth, consistent dependency on $\sigma$.

After training, the model does not simply memorize scores at specific $\sigma$ values, but learns how the score evolves as $\sigma$ changes. This smooth dependency allows us to evaluate the model at arbitrary noise levels and even skip some of them during inference to speed up sampling.

One example of such a method is [DDIM](https://arxiv.org/abs/2010.02502).

From the forward diffusion process (Eq. 4 in DDPM), we know that for any timestep $\tau$:

$$
x_{\tau} = \sqrt{\overline{\alpha_{\tau}}}\, x_0 + \sqrt{1 - \overline{\alpha_{\tau}}}\epsilon, \quad \epsilon \sim \mathcal{N}(0,1).
$$

This holds for all $\tau$.

At inference time, the model predicts the noise $\epsilon_\theta(x_t, t)$, which allows us to estimate the clean image:

$$
\hat{x}_0 =
\frac{1}{\sqrt{\overline{\alpha_t}}}
\left(
x_t - \sqrt{1 - \overline{\alpha_t}}\, \epsilon_\theta(x_t, t)
\right)
$$

This expression is obtained by rearranging the forward equation and replacing the true noise $\epsilon$ with the model prediction.

Since the forward equation is valid for any timestep, we can plug $\hat{x}_0$ back into it to directly compute a sample at an earlier timestep $\tau < t$:

$$
x_{\tau} =
\sqrt{\overline{\alpha_{\tau}}}\, \hat{x}_0
+ \sqrt{1 - \overline{\alpha_{\tau}}} \epsilon, \quad \epsilon \sim \mathcal{N}(0,1)
$$

This allows us to skip intermediate steps and jump across timesteps during inference.

In principle, one could jump directly from $T$ to $0$. However, $\hat{x}_0$ is only an approximation, and large jumps amplify its error, leading to poor sample quality. In practice, we therefore use a reduced but still reasonably dense sequence of timesteps (e.g., 50 instead of 1000, i.e., we take each $\frac{1000}{50}=20$-th noise level: 1000, 980, 960...), achieving a good trade-off between speed and quality.

This approach works because during training, the timestep $t$ is sampled uniformly at random. As a result, the model learns to denoise from any noise level and develops a smooth understanding (given dense-enough discretization, as we discussed above) of how the score depends on $\sigma$, making it possible to skip some of $\sigma$. This enables flexible, non-sequential sampling strategies at inference time.

Let's use DDIM for our Stable Diffusion.

In [ ]:
from diffusers import DDIMScheduler

scheduler = DDIMScheduler.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="scheduler")
scheduler.set_timesteps(50)

Another important aspect to understand is the role of the VAE in latent diffusion models.

Originally, a VAE with encoder $\phi$ and decoder $\theta$ is trained by maximizing the ELBO:
$$
\mathbb{E}_{q_{\phi}(z \mid x)}[\log p_{\theta}(x \mid z)] - \mathrm{KL}(q_{\phi}(z \mid x) \| p(z)),
$$
where the prior is typically chosen as $p(z) = \mathcal{N}(0, I)$.

The KL term can be decomposed as:
$$
\mathbb{E}_{p(x)}\left[\mathrm{KL}(q_{\phi}(z \mid x) \| p(z))\right]
=
\mathrm{KL}(q_{\phi}(z) \| p(z)) + I(x; z),
$$
where $q_{\phi}(z)$ is the aggregated posterior and $I(x; z)$ is the mutual information.

Thus, the objective contains three competing effects. The reconstruction term encourages high mutual information so that $z$ retains enough information to reconstruct $x$. The distribution matching term encourages $q_{\phi}(z)$ to match the prior $\mathcal{N}(0, I)$. At the same time, the KL term also penalizes the mutual information $I(x; z)$, effectively encouraging compression.

We model:
$$
q_{\phi}(z \mid x) = \mathcal{N}(\mu(x), \sigma(x)^2)
$$

Minimizing the KL term encourages $\mu(x)$ to stay close to $0$ and $\sigma(x)$ to stay close to $1$. However, the reconstruction term requires different inputs $x$ to map to distinguishable latent representations. This creates a fundamental tension: the KL term pushes latent codes to overlap, while the reconstruction term pushes them apart.

If the KL term is too strong, many different inputs may map to overlapping latent regions. In that case, the decoder receives ambiguous inputs and learns to predict the conditional expectation:
$$
\hat{x}(z) = \mathbb{E}[x \mid z],
$$
which results in blurry generations.

While enforcing $q_{\phi}(z) \approx \mathcal{N}(0, I)$ allows VAE-driven generation by sampling $z \sim \mathcal{N}(0, I)$, it comes at the cost of reconstruction quality. In latent diffusion models, we do not rely on the VAE for generation. Instead, VAE is used purely as compression and diffusion models learn the latent distribution $q(z)$ directly. Therefore, we prioritize high-quality reconstruction over matching the prior.

This is achieved using the [$\beta$-VAE](https://openreview.net/forum?id=Sy2fzU9gl) formulation:
$$
\mathbb{E}_{q_{\phi}(z \mid x)}[\log p_{\theta}(x \mid z)] - \beta \,\mathrm{KL}(q_{\phi}(z \mid x) \| p(z))
$$

Choosing $\beta \ll 1$ weakens the KL constraint, improves reconstruction quality, and allows latent representations to better separate different inputs.

Using the reparameterization trick:
$$
z = \mu(x) + \sigma(x)\epsilon, \quad \epsilon \sim \mathcal{N}(0,1)
$$

The reconstruction term favors stable latents, which pushes $\sigma(x)$ toward zero. When the KL term is weak, this leads to:
$$
q_{\phi}(z \mid x) \approx \delta(z - \mu(x))
$$

Thus, the encoder becomes nearly deterministic, and the aggregated distribution $q_{\phi}(z)$ is approximately the distribution of $\mu(x)$ over the dataset, which is generally not Gaussian and may have arbitrary variance.

In DDPM, the forward process is:
$$
z_t = \sqrt{\bar{\alpha}_t} z_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)
$$

This process assumes that the signal $z_0$ and noise are on comparable scales. However, with $\beta$-VAE, the latent distribution does not necessarily have unit variance. If the variance of $z_0$ is too large, the noise is ineffective; if it is too small, the signal is destroyed too quickly.

To address this, latents are rescaled:
$$
z' = c \cdot z
$$

where $c$ is chosen such that:
$$
\mathrm{Var}(z') \approx 1
$$

In practice, $c$ is estimated from dataset statistics of $\mu(x)$. Before decoding, the scaling is inverted:
$$
z = \frac{z'}{c}
$$

### Task 1 [0.5 pts]

1. [0.15] Given the CLIP model above, create a function `get_text_embeddings` that will convert a raw text string into textual embeddings.

2. [0.05] Create a `get_latents` function that will create $z_T$ (random noise).

3. [0.10] Create a `decode_latents` function that will take predicted $z_0$, scale it using $c$ set to `vae.config.scaling_factor` and pass through the VAE decoder above.

Note that for stability, images are transformed from $[0,1]$ to $[-1,1]$ to have zero-mean, so your `decode_latents` needs to account for that.

4. [0.20] Create a `denoise_latents` function that will go over $t$ from the `scheduler` and iteratively denoise $z_t$ using `scheduler.step` and `unet` (see `diffusers` documentation). Initialize $z_t$ with $z_T$ and do not forget about the text conditioning.


In [ ]:
# YOUR CODE HERE

Run your functions with the prompt from the start of the notebook and show the result here:

In [ ]:
# YOUR CODE HERE

### Classifier-Free Guidance Experiments

In generative models, there is a well-known concept of *vector arithmetic*. This was first observed in GANs in [Unsupervised Representation Learning with Deep Convolutional Generative Adversarial Networks (Section 6.3.2)](https://arxiv.org/abs/1511.06434). The idea is that if one noise vector corresponds to object $x_1$ and another to object $x_2$, then averaging these vectors produces something resembling a mixture of both. More generally, one can modify an object by moving its latent representation along a “direction” corresponding to a specific feature or style.

In diffusion models, when we predict noise $\epsilon$ using a U-Net conditioned on text, we obtain $\epsilon_{\text{text}}$. This prediction implicitly contains two types of information:

1. General image structure (what makes an image look plausible), independent of the prompt  
2. Prompt-specific information (what makes the image match the text, e.g., "a house")

We can also compute an unconditional prediction $\epsilon_{\text{uncond}}$ by using an empty prompt. This captures only the general, non-prompt-specific information.

The difference
$$
\epsilon_{\text{text}} - \epsilon_{\text{uncond}}
$$
can be interpreted as a *direction* in the latent space that pushes the prediction toward the prompt.

Using this, we define:
$$
\epsilon_{\text{total}} = \epsilon_{\text{uncond}} + s \cdot (\epsilon_{\text{text}} - \epsilon_{\text{uncond}})
$$

where $s$ is the **guidance scale**:
- $s = 0$: purely unconditional generation  
- $s = 1$: standard conditional prediction  
- $s > 1$: stronger alignment with the prompt  

This method is known as *classifier-free guidance*. See [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598) for more details.

Let's now rewrite the code from Task 1 using this formulation and explore how different values of $s$ affect generation.

### Task 2 [0.5 pts]

[0.4] Create a `denoise_latents_cfg` function that will compute $\epsilon_{\text{total}}$ for a provided scale $s$.

[0.1] Think how to compute $\epsilon_{\text{total}}$ in a single forward pass and write your function in the corresponding way.

In [ ]:
# YOUR CODE HERE

Show the result for $s=0, s=1, s=10$.

In [ ]:
# YOUR CODE HERE

### Task 3 [1 pts]

Your task is to understand and analyze the impact of the scaling factor $s$. To do so, you should run the classifier-free guidance for different $s$. Think of a reasonable set of possible $s$ options that will allow you to understand the affect of $s$ fully (think about reasonable cases, meaningful cases, and corner cases). Think about a nice visualization of your experiments.

In [ ]:
# YOUR CODE HERE

Reflect on the generated images for each $s$. How different values of $s$ impact the generation?

**YOUR RESPONSE HERE:**

## Part 2: Implementing a Diffusion Model

**Note** Everything from the `diffusers` library is banned in this part, except for VAE. **Solutions from the internet/LLM are also banned**.

In this part, we will go through [Denoising Diffusion Probabilistic Models (DDPM)](https://arxiv.org/abs/2006.11239) and implement it. Our final goal is to create a Latent Diffusion Model that will be able to generate high-resolution images. However, diving straight into this task is too complicated and it is not how one should approach a paper re-implementation or creation of a pipeline (training/inference) from scratch in general. We need to start with something simpler and build the final model step-by-step. Here is our plan:

1. Instead of complicated images, we will start with a toy dataset. Since the dataset is simple, a correct (or at least "more or less" correct) pipeline will be able to work on it without issues. If our model fails on it, then, we definitely did a mistake somewhere and need to debug before moving on with a more complicated dataset.
2. As you remember, in the diffusion process, a model is trained to predict added noise. Even though current SOTA models use [U-Net](https://arxiv.org/abs/1505.04597) or [ViT](https://arxiv.org/abs/2010.11929) backbones, the process itself does not actually require any specific architecture. Hence, before implementing a complicated model, we should first check our pipeline (diffusion, its inference and training) with a simple architecture, such as an MLP, and only if it works, we can move on with a U-Net or something else.
3. Once we make sure that we can do proper generations on a toy dataset with an MLP-based diffusion, we will implement U-Net and run diffusion on high-resolution images.

For the toy dataset, we will use the Moon Dataset from `sklearn`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format='retina'

from sklearn.datasets import make_moons

X, y = make_moons(n_samples=1000, noise=0.1, random_state=1)

plt.scatter(X[y==0, 0], X[y==0, 1], label="Class 0", alpha=0.7)
plt.scatter(X[y==1, 0], X[y==1, 1], label="Class 1", alpha=0.7)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()

plt.show()

We aim to build a Diffusion class that progressively injects noise into the moon-shaped data (forward process) and then learns to reverse this by removing the noise (backward process). To train this model, we need suitable training data, so we'll construct a simple `PyTorch` dataset. For better numerical stability, the data will be normalized to have a mean of zero and a standard deviation of one.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

class MoonDataset(Dataset):

    def __init__(self, n=200):
        X, y = make_moons(n_samples=n, noise=0.1, random_state=1)
        self.X = torch.tensor(X).to(torch.float32)
        self.y = torch.tensor(y)
        self.n = n

        self.mean = self.X.mean(0)
        self.std = self.X.std(0)


    def __len__(self):
        return self.n

    def __getitem__(self, index):
        X = self.X[index]
        # normalize
        X = (X - self.mean) / self.std
        y = self.y[index]
        return X, y

In [ ]:
dataset = MoonDataset(n=10000)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

### Task 4 [2 pts]

Now, we will implement the DDPM-style diffusion. To do so, read the [paper](https://arxiv.org/abs/2006.11239) and fill-in the gaps in the skelet of the Diffusion class below. See doc-strings for hints.


*Hint*: For improved numerical stability, it is recommended to use $\log \text{Var}$ instead of $\text{Var}$. This is especially helpful for small $\sigma$, as it provides a better numerical range, more stable gradients, and ensures positivity:

$$
\sigma^2 = \exp(\log \text{Var})
$$

**Note**: Ignore class labels for now.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class GaussianDiffusion(nn.Module):
    """
    DDPM Implementation
    """
    def __init__(self, model, image_shape, timesteps=1000, beta_schedule="linear", device=None):
        super().__init__()
        self.model = model
        self.image_shape = image_shape  # e.g. (C,H,W)
        self.T = timesteps
        self.device = device or next(model.parameters()).device

        if beta_schedule == "linear":
            betas = torch.linspace(1e-4, 2e-2, self.T) # See Section 4
        else:
            raise ValueError("Unknown beta_schedule")

        # register buffers
        # we pre-compute common calculations, such as alpha, \overline{alpha}, etc.
        self.register_buffer("betas", # YOUR_CODE_HERE)
        alphas = # YOUR_CODE_HERE
        alphas_cumprod = # YOUR_CODE_HERE
        alphas_cumprod_prev = # YOUR_CODE_HERE

        self.register_buffer("alphas", # YOUR_CODE_HERE)
        self.register_buffer("alphas_cumprod", # YOUR_CODE_HERE)
        self.register_buffer("alphas_cumprod_prev", # YOUR_CODE_HERE)

        self.register_buffer("sqrt_alphas_cumprod", # YOUR_CODE_HERE)
        self.register_buffer("sqrt_one_minus_alphas_cumprod", # YOUR_CODE_HERE)
        self.register_buffer("sqrt_recip_alphas", # YOUR_CODE_HERE)

        # posterior q(x_{t-1} | x_t, x_0)
        posterior_variance = # YOUR_CODE_HERE
        # clip variance with a very small value (1e-20) before taking log for safety
        posterior_log_variance_clipped = # YOUR_CODE_HERE
        posterior_mean_coef1 = # YOUR_CODE_HERE
        posterior_mean_coef2 = # YOUR_CODE_HERE

        self.register_buffer("posterior_variance", # YOUR_CODE_HERE)
        self.register_buffer("posterior_log_variance_clipped", # YOUR_CODE_HERE)
        self.register_buffer("posterior_mean_coef1", # YOUR_CODE_HERE)
        self.register_buffer("posterior_mean_coef2", # YOUR_CODE_HERE)

    def extract(self, a, t, x_shape):
        """
        Extract element at timestamp t from tensor a.

        Args:
          a (Tensor): timestep-dependent tensor (T, )
          t (LongTensor): timestep (B, )
          x_shape (torch.Size): size of x
        Returns:
          out (Tensor): (B, 1, 1, 1...) broadcasted tensor.
            i-th row containts t[i]-th row from a.
        """
        return # YOUR_CODE_HERE

    @torch.no_grad()
    def q_sample(self, x0, t, noise=None):
        """
        Sample x_t from q(x_t | x_0).
        Eq. 4 from the paper.

        Args:
          x0 (Tensor): start.
          t (LongTensor): timestep (B, )
          noise (Tensor | None): optional noise.
        Returns:
          xt (Tensor): sample from q(x_t | x_0)
        """
        return # YOUR_CODE_HERE

    def q_sample_loop(self, x0, log_step=100):
        """
        Apply q_sample sequentially.
        Also returns intermediate representations.

        Args:
          x0 (Tensor): start.
          log_step (int): return k*log_step-th representation.
        Returns:
          xt (Tensor): final noise.
          x_inter (list[Tensor]): intermediate representations.
        """
        x_inter = []
        xt = x0 # start with x0
        for i in range(self.T):
            # YOUR_CODE_HERE
            if i % log_step == 0:
                x_inter.append(xt)
        # append final representation to x_inter
        # YOUR_CODE_HERE
        return xt, x_inter  # typically clamp/convert to [0,1] outside

    def q_posterior(self, x0, xt, t):
        """
        Compite mean, var, and log_var
        for q(x_{t-1} | x_t, x_0).
        Eq. 6 from the paper.

        Args:
          x0 (Tensor): start.
          xt (Tensor): sample after t additions of noise.
          t (LongTensor): timestep (B, )
        Returns:
          mean (Tensor): mean.
          var (Tensor): var.
          log_var (Tensor): log_var.
        """
        return # YOUR_CODE_HERE


    def p_mean_variance(self, xt, t):
        """
        Compute mean from Eq 11 and variance.
        We use \tilde{\beta} version of variance, a.k.a. fixed-small
        (i.e. the same variance as in q)

        Args:
          xt (Tensor): sample after t additions of noise.
          t (LongTensor): timestep (B, )
        Returns:
          model_mean (Tensor): computed mean.
          posterior_var (Tensor): computer var.
          posterior_log_var (Tensor): computed log_var
        """
        return # YOUR CODE HERE

    @torch.no_grad()
    def p_sample(self, xt, t):
        """
        Sample x_{t-1} from xt and t.

        Combines Eq.1 and p_mean_variance

        Args:
          xt (Tensor): current noisy sample.
          t (LongTensor): (B, ) which timestep to use to get x_{t-1}
        Returns:
          x_{t-1} (Tensor): previous sample.
        """
        # be careful with t==0
        return # YOUR_CODE_HERE

    @torch.no_grad()
    def sample(self, n, log_step=100):
        """
        Sample n images from random noise.

        Args:
          n (int): number of images.
          log_step (int): return k*log_step-th representation.
        Returns:
          x0 (Tensor): x0 from random noise.
          x_inter (list[Tensor]): intermediate representations.
        """
        # YOUR_CODE_HERE
        return xt, x_inter  # typically clamp/convert to [0,1] outside

    def compute_loss(self, x0):
        """
        Compute loss (Eq. 14)

        Args:
          x0 (Tensor): start position.
        """
        return # YOUR_CODE_HERE


### Task 5 [0.25 pts]

To estimate the added noise, we need a neural network that can approximate $\epsilon$ from the current noisy sample $x_t$ and the timestep $t$. The original DDPM model uses a U-Net architecture, but since our toy dataset is quite simple, a fully connected network (MLP) is enough. This will also allow us to test our pipeline without the complexity of a more advanced architecture implementation.

Design a basic MLP consisting of four Linear layers. Additionally, include timestep embeddings (using `nn.Embedding`) to represent $t$, and concatenate these embeddings with the input before feeding everything into the network.

In [ ]:
# YOUR CODE HERE

### Task 6 [0.5 pts]

Create a training loop for your diffusion model and train it on the moon dataset. Use $T=1000$ timesteps for noise scheduling.

**Note**: In diffusion models, it's common for the loss to plateau after a certain point, yet the sample quality can continue to improve. Therefore, it's important to track additional metrics and periodically generate samples during training to assess whether the outputs are getting better. Make sure to log some generated samples throughout the training process (but we do not expect you to have any metrics at this point, only samples).

In [ ]:
# YOUR CODE HERE

Now we can sample noise from $\mathcal{N}(0,1)$ and pass it through the diffusion to generate some moon object samples.

Sample 1000 examples and visualize them.

In [ ]:
# YOUR CODE HERE

### Task 7 [0.25 pts]

If you followed the previous steps correctly, your generations should look similar to the moons. However, we haven't used classes yet. Let's introduce them now. We only need to update the model (so it gets class label as input as well) and the section of the diffusion class where we pass inputs to the model.

$$
\epsilon_{\theta}(x_t, t, y), \text{where $y$ is the class}
$$

In [ ]:
# YOUR CODE HERE

For classes, we will again use `nn.Embedding` and just concatenate it with other input before passing to the MLP. Create an updated class:

In [ ]:
# YOUR CODE HERE

Train a conditional version of your model:

In [ ]:
# YOUR CODE HERE

### Task 8 [0.5 pts]

Let's sample from this Conditional Diffusion. Sample random classes uniformly and then samples for each class. Visualize the results:

In [ ]:
# YOUR CODE HERE

Again, we get moon-like shapes, as we expected. However, we see that class conditioning also produced sharper moons. Why?

**YOUR RESPONSE HERE**:

### Task 9 [0.5 pts]

Let's see how $x_t$ changes over time in the forward and reverse processes.

For the forward process, we will provide moons from the dataset as $x_0$. Compute final and intermediate representations (each 100-th $t$). Do-not forget about denormalization.

In [ ]:
# YOUR CODE HERE

Now, show more steps between 0 and 200 to get a fine-grained understanding of what is happening with the $x_t$.

In [ ]:
# YOUR CODE HERE

What can you say about the connection between $t$ and $x_t$? How the similarity between moons and $x_t$ changes with $t$?

**YOUR RESPONSE HERE:**

### Task 10 [0.75 pts]

Now that we know that our `Diffusion` module is correct by testing it on a simple task with a simple architecture, we can move on with a more advanced architecture and a real-life problem.

We will focus on the super-resolution task, i.e., given a coarse image $c$, we want to restore its detailed version $x$.
$$
\text{DDPM}(x_{T}, c) \to x
$$

Since the images are large, we will work in the LDM setting, i.e., we will apply diffusion in the latent VAE space instead of a pixel space.

First, we need to get a dataset.

1. Download [CelebHQ](https://huggingface.co/datasets/mattymchen/celeba-hq) dataset.
2. Downscale images to $128 \times 128$ **using `bicubic` method**. This will be our target $x$.
3. Downscale images further to $16 \times 16$  **using `bicubic` method** and then upscale this low-resolution image back to $128 \times 128$  **using `bicubic` method**. This is our coarse conditional image $c$.
4. Use VAE from Stable Diffusion (the one we used in the First Part of the Homework) and encode $x$ and $c$ into the latent space. Use $\mu(x)$ without sampling for deterministic results.

**Hint:** you can do it using HF `.map` batched operation to save RAM.

**Hint:** you may want to save the dataset on disk or upload it to your HF account to avoid rerunning this step every time you re-run the session.

**Note** even though VAE is pretrained for even higher resolution images, it still works well for our $128 \times 128$ case.

In [ ]:
# YOUR CODE HERE

Create a `PyTorch` dataset that will return the target and condition images in the latent space. Create a DataLoader. Use batch size set to $32$

In [ ]:
# YOUR CODE HERE

### Task 11 [1 pts]

For estimating noise, we will use the UNet architecture with the following setup:

1. Each level consists of

    * Two ResBlocks with 2 convolutions of $3\times 3$ size, BatchNorm, and GELU.
    * Average Pooling for Downsampling / ConvTranspose for Upsampling

2. Three downsample/upsample blocks with the Self-Attention in the last low-level blocks.

3. 1 Mid Block with ResBlock + Self-Attetion + ResBlock.

4. Each level gets twice as more chanells when going down, going up is symmetric.

5. Time conditioning is applied by projecting a temporal embedding into each of the ResBlocks' spaces and adding it between two convolutions.


While there are plenty of ways how conditioning can be designed for UNet, we will use the most simple approach and concatenate $z_t$ and $c$ across channels in the UNet input.

**Note:** You must write the UNet from scratch yourself. You are only allowed to use simple `nn` layers, such as Linear, Convolution (+ Transposed), Activation layers, Embedding, Normalization Layers, and Self-Attention, `nn.Sequential` or `nn.ModuleList`.

In [ ]:
# YOUR CODE HERE

### Task 12 [0.75 pts]

A common metric for evaluating generative models is [FID](https://en.wikipedia.org/wiki/Fr%C3%A9chet_inception_distance), which compares the distributions of the real and generated data. However, when we know the matching between real and generated samples, we can use more interpretable and informative metrics.

In super-resolution, we know the ground-truth high-resolution image $x$ for each condition $c$. Therefore, we can compare directly $x$ with the generated $\hat{x}$. We care about three scores:

* Pixel Similarity

* Structural Similarity

* Perceptual Similarity

PSNR, SSIM, and LPIPS are the metrics that give us these three measurements.

Write the code that will run your diffusion model over the validation set and compute these metrics.

**Note**: you can use `torchmetrics` and `lpips` libraries. For LPIPS, use the `alexnet` (default) variant.

In [ ]:
# YOUR CODE HERE

### Task 13 [1.5 pts]

Now create the training script. Log metrics and sample some images each epoch. In total, run these three experiments:

1. **One-Batch Test**. This is another technique for validating your pipeline before moving on with the full training. Given a fixed batch $B$, you train your model only on this $B$ for many steps **and evaluate also on the same batch $B$**. If your pipeline is correct, the model should overfit on this batch, i.e., achieve good metrics, low loss, and nice samples. Depending on the task, the loss and metrics can achieve the perfect scores; but for diffusion, we will primary look at the generated samples and how well they match the target image. For this experiment, use $T=50$. **Important**: passing one-batch test *does not gurantee* that your pipeline is fully correct, but not passing it means you have a bug somewhere.

2. Run full dataset training with $T=100$ diffusion steps in the noise schedule.

3. The same as 2, but with $T=1000$.

**Note**: during training, you can evaluate only on a subset.

**Note**: to get the full grade, your $T=100$ model must be able to achieve $\text{LPIPS} < 20.5$

In [ ]:
# YOUR CODE HERE

Evaluate the performance of model with $T=100$ and with $T=1000$ on the full validation dataset. Compare with the metrics for the `bicubic` baseline (the $c$ itself).

In [ ]:
# YOUR CODE HERE

What can you say about PSNR/SSIM/LPIPS metrics for bicubic and diffusion? Are the results surprising? Explain the differences you see and if the result is reasonable. Also, what difference do you see between $T=100$ and $T=1000$? Why is it like that? What are the pros and cons of changing $T$?

**Hint** look at the definition of the metrics.

**YOUR RESPONSE HERE:**

### Bonus [1.5 pts]

Implement deterministic [DDIM](https://arxiv.org/abs/2010.02502) version of inference for your model. We discussed the core idea in the first part of the notebook.

**Note** DDIM is a sampling inference technique, you do not need to retrain the model.

In [ ]:
# YOUR CODE HERE

Compare DDIM with different number of steps $N$ vs DDPM in terms of generation quality and speed. Similarly to Classifier-Free Guidance experiments, we expect you to come up with a meaningful and informative experiment fully exploring the trade-off between DDIM and DDPM. Think about the experiment design and presentation options for your results and analysis. We will evaluate both the experimental setup and the analysis.

**Hint** recall how we designed experiments during the course, what concepts we introduced, and what advice we gave.

In [ ]:
# YOUR CODE FOR EXPERIMENTS

**YOUR ANALYSIS OF THE EXPERIMENTS HERE:**